### 🚀 Run this notebook in [RenkuLab](https://renkulab.io/)

👉 [![launch - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/meteoswiss/opendata-nwp-demos/sessions/01KME52HC2FZ6ZHB30SSFG08PW/start)

1. Wait for the session to start
2. Navigate to `opendata-nwp-demos/10_pollen_conversion.ipynb` and run the notebook without any further installation.

# Retrieving, Converting, and Visualizing ICON-CH2-EPS Pollen Forecast Data

The numerical weather forecast model ICON can also provide pollen forecasts. All relevant input is considered, such as the distribution of plants, the actual flowering state and the weather. In addition, the most recent observations of MeteoSwiss' real-time pollen observation network are used to fine-tune the pollen forecasts.

At MeteoSwiss, all control runs of ICON-CH2-EPS (initialized at  00, 06, 12 and 18h UTC) calculate pollen concentrations all up to +120 hours ahead. Depending on the season, species include hazel, alder, birch, grass and Ambrosia pollen. These forecasts are important information for allergy sufferers to plan their medication and leisure time to minimize exposure. Roughly 20% of the Swiss population is affected.

This notebook demonstrates the full workflow for accessing, converting, and visualizing determinstic ICON-CH2-EPS pollen forecast data. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).

The ICON model outputs pollen forecasts as specific number concentration — the number of particles per kg of air. This pollen forecast could be used "as is". However, pollen observations are usually provided as volume concentrations (number of particles per volume air). If pollen forecasts should be comparable with observations, the forecasts need to be converted to volume concentrations. This is achieved by simply multiplying the specific number concentrations with the density of the air. 

In this notebook this conversion is demonstrated, either by using model data or with a significantly cheaper approximation.

For visualization, this notebook uses the [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/examples/guide/01-introduction.html) library developed by ECMWF, which offers intuitive plotting tools for meteorological data.

---

## 🔍 **What This Notebook Covers:**

 📥  **Retrieve**  
    Fetch deterministic ICON-CH2-EPS forecast data (pollen and variables required for conversion to volume concentration) via the `ogd_api` module.

 📐  **Convert to volume concentration**  
    Convert specific pollen number concentration to volume concentration using model data with [meteodata-lab](https://meteoswiss.github.io/meteodata-lab/) operators, or with a simple approximation.
 

 🧭  **Regrid**  
    Interpolate ICON-CH2-EPS data from its native, icosahedral grid to the regular latitude/longitude grid [WGS84 (EPSG:4326)](https://epsg.io/4326).

 🌍  **Visualize**  
    Plot the processed data on a map using [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/examples/guide/01-introduction.html).

---

## Retrieving Pollen Forecast
In this first part, we retrieve deterministic ICON-CH2-EPS pollen forecast data. To access this data, we use the `ogd_api` module from the [meteodata-lab](https://meteoswiss.github.io/meteodata-lab/) library — a convenient interface for accessing numerical weather forecasts via the [STAC (SpatioTemporal Asset Catalog) API](https://data.geo.admin.ch/api/stac/static/spec/v1/apitransactional.html#tag/Data/operation/getAsset), which provides structured access to Switzerland’s open geospatial data.

## 📥 Retrieve ICON-CH2-EPS Pollen Data

We request the pollen variable `TODO`, in addition to temperature `T`, pressure `P`, specific humidity `QV` and the cloud mixing ratio `QC`. Those additional variables will later be used to compute the total air density which is needed for conversion to the volume number density.

>⏰ **Forecast Availability**: Forecast data will typically be available a couple of hours after the reference time — due to the model runtime and subsequent upload time. The data remains accessible for 24 hours after upload.

### ⏰ Pollen Data Availability
Pollen data is only available during the pollen season, different for each pollen species. On the first day, pollen are initialized in the KENDA-CH1 08:00 UTC cycle and available in ICON-CH2-EPS starting from the 12:00 UTC run.

| species            | short name    | first day of year | last day of year |
| ------------------ | ------------- | ----------------- | ---------------- |
| alder (alnus)      | ALNU          | 8                 | 90               |
| ragweed (ambrosia) | AMBR          | 190               | 273              |
| birch (betula)     | BETU          | 77                | 145              |
| hazel (corylus)    | CORY          | 8                 | 76               |
| grasses (poaceae)  | POAC          | 91                | 243              |


In [ ]:
from meteodatalab import ogd_api

param_list = ['DEN', 'POAC']
reqlist = []

for param in param_list:
    req = ogd_api.Request(
        collection="ogd-forecasting-icon-ch2",
        variable=param,
        ref_time="latest",
        perturbed=False,
        lead_time="P0DT6H",
    )
    reqlist.append((param, req))

> 💡 **Tip**: Use temporary caching with earthkit-data to skip repeated downloads — it's auto-cleaned after the session.
> *For more details, see the [earthkit-data caching docs](https://earthkit-data.readthedocs.io/en/latest/examples/cache.html)*.

> 💡 **Hint**: If you get an error message containing `HTTPError: 403 Client Error: Forbidden for url`, you may be trying to retrieve data older than 24h hours! Please adjust your requests.

In [ ]:
from earthkit.data import config
config.set("cache-policy", "temporary")

da_dict = {}
for param, req in reqlist:
    da = ogd_api.get_from_ogd(req)
    da_dict[param] = da

## 📐 Convert to volume concentration

The volume number concentration is obtained by multiplying the specific number concentration by the total air density $\rho$.

> ⚠️ **Location of pollen variables**: On the grid, the provided pollen variables and density DEN are associated with the lowest full level, i.e., represent the mean value over the first 20m above ground.

In [ ]:
import xarray as xr

# conversion to volume number concentration: multiply with density
with xr.set_options(keep_attrs=True):
    # TODO: fix units
    poac_concentration = da_dict['POAC'] * da_dict['DEN']

## 🗺️ Visualize the data

In [ ]:
# Define regridding
from rasterio.crs import CRS
from meteodatalab.operators import regrid

# Define ~1 km target grid over ICON-CH1-EPS domain
extent = (-0.817, 18.183, 41.183, 51.183)  # (xmin, xmax, ymin, ymax)
nx, ny = 732, 557

# Create regular lat/lon grid (EPSG:4326)
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)

In [ ]:
# regrid
poac_reg = regrid.iconremap(poac_concentration, destination)

In [ ]:
import pandas as pd
import cartopy.crs as ccrs
from earthkit.plots.geo import bounds, domains
import earthkit
from earthkit.plots.styles import Style
import numpy as np


# === Setup Domain (Switzerland) ===
bbox = bounds.BoundingBox(5.7, 10.5, 45.6, 48, ccrs.Geodetic())
domain = domains.Domain.from_bbox(bbox=bbox)


# === Create Map ===
chart = earthkit.plots.Map(domain=domain)

#levels = np.linspace(0.8, 1.3, 11)

contourf_style = Style(
    #levels=levels,
    colors="YlGnBu",
    legend_style="colorbar",
)
chart.contourf(poac_reg, x="lon", y="lat", style=contourf_style)


# === Add Map Features ===
chart.land()
chart.coastlines()
chart.borders()
chart.gridlines()


# === Annotate Chart ===
ref_time = pd.to_datetime(poac_reg.coords["ref_time"].values[0]).strftime("%Y-%m-%d %H:%M UTC")
lead_time = poac_reg.coords["lead_time"].values.astype('timedelta64[h]')

parameter = poac_reg.attrs["parameter"]
title = f"TODO Title | {ref_time} (+{lead_time})"
legend_label = f"{parameter['name']} ({'TODO unit'})"

chart.title(title)
chart.legend(label=legend_label)


# === Show Plot ===
chart.show()
